### Libraries

In [ ]:
from anndata import AnnData
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import scanpy as sc
from sklearn.preprocessing import OneHotEncoder
import torch
from tqdm import tqdm

from sc_exp_design.config import NeuralVelocityFieldConfig
from sc_exp_design.models import FlowMatching

# defining figure size
FIGSIZE = (3, 3)
rcParams["figure.figsize"] = FIGSIZE


### Reading data

In [ ]:
PATH = "../../theis/HID01_fcs_concatenated.h5ad"
adata = sc.read_h5ad(PATH)
adata


### Subsampling

In [ ]:
# optionally subsample the data
KEEP_COPY = True
SUBSAMPLE = True
SUBSAMPLING_DONE = False
SUBSAMPLE_SIZE = 1e-2

if SUBSAMPLE and not SUBSAMPLING_DONE:
    if KEEP_COPY:
        adata_original = adata.copy()
        sc.pp.subsample(adata, SUBSAMPLE_SIZE)
    SUBSAMPLING_DONE = True
adata


### Applying transformation

In [ ]:
cofactor = 100
key_arcsin = f"X_arcsinh_cof{cofactor}" 
adata.obsm[key_arcsin] = np.arcsinh(adata.X/cofactor)
adata


### One hot encode conditions.

In [ ]:
EXPERIMENTAL_COLUMNS = [
    'mtg_[µm]', 'rhflt3l_[ng/ml]', 'gm-csf_[ng/ml]', 'tpo_[ng/ml]', 'sr1_[nm]',
    'um171_[nm]', 'um729_[µm]', 'scf_[ng/ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]',
    'ldl_[ng/ml]', 'il3_[ng/ml]', 'o2_[%]',
]
unique_conds = list(adata_original.obs.groupby(by=EXPERIMENTAL_COLUMNS).groups.keys())

unique_conds_str = ["_".join([str(c) for c in cond]) for cond in unique_conds]
adata.obs["protocol"] = adata.obs[EXPERIMENTAL_COLUMNS].astype(str).apply(lambda x: '-'.join(x), axis=1)

ohe = OneHotEncoder()
ohe.fit(adata.obs["protocol"].values.reshape(-1, 1))

one_hot_dict = {}
for cat in adata.obs["protocol"].unique():
    dataset_onehot = (
        ohe.transform(np.array([cat]).reshape(-1, 1)).toarray().flatten()
    )
    one_hot_dict[cat] = dataset_onehot
adata.uns["protocol_one_hot"] = one_hot_dict
adata


### Initializing flow model.

In [ ]:
flow_matching = FlowMatching(
    coupling_type="independent",
    generate_from_noise=True,
)


### Preparing data.

In [ ]:
flow_matching.prepare_train_data(
    adata,
    sample_rep=key_arcsin,
    perturbations=("protocol", ),
    perturbation_reps={
        "protocol": ("protocol_one_hot",),
    },
)


### Initialize velocity field.

In [ ]:
cvf_config = NeuralVelocityFieldConfig(
    flow_dim=adata.obsm[key_arcsin].shape[1],
    encode_state=True,
    state_encoder_output_dim=64,
    state_encoder_mlp_kwargs={
        "hidden_dims": (64, 64),
        "use_batchnorm": False,
        "use_dropout": False,
        "dropout_rate": 0.0,
        "activation_class": torch.nn.SELU,
        "final_activation_class": torch.nn.Identity,
    },
    use_guidance=True,
    encode_conditions=True,
    perturbation_layers_before_pooling={
        "repr_protocol_protocol_one_hot":{
            "input_dim": list(one_hot_dict.values())[0].shape[0],
            "output_dim": 16,
            "hidden_dims": (64, ),
            "use_batchnorm": False,
            "use_dropout": False,
            "dropout_rate": 0.0,
            "activation_class": torch.nn.SELU,
            "final_activation_class": torch.nn.Identity,
        }
    },
    perturbation_layers_after_pooling={
        "hidden_dims": (64, ),
        "use_batchnorm": False,
        "use_dropout": False,
        "dropout_rate": 0.0,
        "activation_class": torch.nn.SELU,
        "final_activation_class": torch.nn.Identity,
    },
    decoder_mlp_kwargs={
        "hidden_dims": (64, 64),
        "use_batchnorm": False,
        "use_dropout": False,
        "dropout_rate": 0.0,
        "activation_class": torch.nn.SELU,
        "final_activation_class": torch.nn.Identity,
    },
)
flow_matching.prepare_model(
    cvf_config,
    num_time_steps=100,
)


### Training the model.

In [ ]:
flow_matching.train(
    num_training_steps=20_000,
)
flow_matching.trainer.plot_training_logs()


### Sampling some events for each protocol condition.

In [ ]:
out_dict = {}
n_samples_per_protocol = 10_000

for protocol_id, protocol_one_hot in tqdm(adata.uns["protocol_one_hot"].items()):
    cond = torch.from_numpy(protocol_one_hot).repeat(n_samples_per_protocol, 1).float().to(flow_matching.device)
    samples = flow_matching.predict(
        batch={
            "condition": {
                "repr_protocol_protocol_one_hot": cond
            }
        },
    )
    out_dict[protocol_id] = samples.detach().cpu().numpy()


In [ ]:
def concatenate_adata_and_plot(
    adata, samples, plot_pcs=True, plot_umap=True, subsample_size=0.1, coloring=["dataset_type"], key="X_pca", **kwargs
):
    obs = {"dataset_type": ["Real" for _ in range(adata.shape[0])] + ["Generated" for _ in range(samples.shape[0])], **kwargs}
    adata_generated = AnnData(X=np.concatenate([adata.obsm[key], samples], axis=0), obs=obs)
    adata_generated.obsm[key] = adata_generated.X.copy()

    if plot_umap:
        sc.pp.subsample(adata_generated, subsample_size)
        sc.pp.neighbors(adata_generated)
        sc.tl.umap(adata_generated)
        
    for color in coloring:
        if plot_pcs:
            sc.pl.embedding(adata_generated, basis=key, color=color)
        if plot_umap:
            sc.pl.umap(adata_generated, color=color)
    return adata_generated
